In [ ]:
pip install -U pandas numpy tqdm scikit-learn transformers torch sentencepiece sentence-transformers hdbscan umap-learn shap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 88.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 92.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 102.6 MB/s eta 0:00:00
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.6.1
    Uninstalling scikit-learn-1.6.1:
      Successfully uninstalled scikit-learn-1.6.1
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googl

In [ ]:
from google.colab import drive; drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!mkdir -p ~/.data
!cp -r /content/drive/MyDrive/data /content/data

In [ ]:
import json
import re
from pathlib import Path
from typing import Any, Dict, List, Optional

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer
import hdbscan
import umap
from sklearn.feature_extraction.text import TfidfVectorizer

DATA_SOURCES = {
    "banki": "/content/data/reviews.json",
    "sravni": "/content/data/sravni_reviews.json",
    "otzovik": "/content/data/otzovik_detailed_reviews.json",
}

OUT_DIR = Path("/content/outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

def get_best_device():
    if torch.cuda.is_available():
        return "cuda", 0
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return "mps",
    return "cpu", -1

TORCH_DEVICE, HF_DEVICE = get_best_device()
print("TORCH_DEVICE:", TORCH_DEVICE, "| HF_DEVICE:", HF_DEVICE)

TORCH_DEVICE: cpu | HF_DEVICE: -1


In [ ]:
TEXT_KEYS_CANDIDATES = ["текст", "отзыв", "содержание", "сообщение",
    "комментарий", "описание"
]
DATE_KEYS_CANDIDATES = ["дата", "дата_отзыва", "дата публикации"]
RATING_KEYS_CANDIDATES = ["оценка", "рейтинг"]

def _read_json_any(path: str) -> Any:
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

def _flatten_records(obj: Any) -> List[Dict[str, Any]]:
    if obj is None:
        return []
    if isinstance(obj, list):
        recs = []
        for x in obj:
            recs.append(x if isinstance(x, dict) else {"value": x})
        return recs

    if isinstance(obj, dict):
        for k in ["data", "reviews", "items", "results"]:
            if k in obj and isinstance(obj[k], list):
                return [r if isinstance(r, dict) else {"value": r} for r in obj[k]]
        for v in obj.values():
            if isinstance(v, list):
                return [r if isinstance(r, dict) else {"value": r} for r in v]
        return [obj]

    return [{"value": obj}]

def _pick_first_existing(d: Dict[str, Any], keys: List[str]) -> Optional[Any]:
    for k in keys:
        if k in d and d[k] not in (None, "", [], {}):
            return d[k]
    return None

def _join_text_fields(rec: Dict[str, Any]) -> str:
    main = _pick_first_existing(rec, TEXT_KEYS_CANDIDATES)
    parts = []
    if isinstance(main, str):
        parts.append(main)
    elif isinstance(main, list):
        parts.extend([str(x) for x in main if x is not None])
    elif main is not None:
        parts.append(str(main))

    for k in ["достоинства", "недостатки", "плюсы", "минусы"]:
        v = rec.get(k)
        if isinstance(v, str) and v.strip():
            parts.append(f"{k}: {v}")

    if not parts:
        for k, v in rec.items():
            if isinstance(v, str) and len(v.strip()) > 20:
                parts.append(v)

    return "\n".join([p.strip() for p in parts if p and p.strip()])

def load_source(name: str, path: str) -> pd.DataFrame:
    raw = _read_json_any(path)
    recs = _flatten_records(raw)

    rows = []
    for i, r in enumerate(recs):
        if not isinstance(r, dict):
            r = {"value": r}

        text = _join_text_fields(r)
        if not text or len(text.strip()) < 10:
            continue

        date = _pick_first_existing(r, DATE_KEYS_CANDIDATES)
        rating = _pick_first_existing(r, RATING_KEYS_CANDIDATES)

        rows.append({
            "source": name,
            "source_id": r.get("id", r.get("_id", i)),
            "date_raw": date,
            "rating_raw": rating,
            "text_raw": text
        })

    return pd.DataFrame(rows)

In [ ]:
URL_RE = re.compile(r"https?://\S+|www\.\S+", re.IGNORECASE)
MULTISPACE_RE = re.compile(r"\s+")
NONPRINT_RE = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")

def clean_text(t: str) -> str:
    t = t.replace("\u00A0", " ")
    t = URL_RE.sub(" ", t)
    t = NONPRINT_RE.sub(" ", t)
    t = t.strip()
    t = MULTISPACE_RE.sub(" ", t)
    return t

In [ ]:
dfs = []
for name, path in DATA_SOURCES.items():
    df = load_source(name, path)
    dfs.append(df)

df_all = pd.concat(dfs, ignore_index=True)
df_all["text"] = df_all["text_raw"].map(clean_text)
df_all = df_all[df_all["text"].str.len() >= 10].reset_index(drop=True)

df_all.head(3)

,source,source_id,date_raw,rating_raw,text_raw,text
0,banki,12906258,None,None,https://www.banki.ru/services/responses/bank/r...,"Оформлял дебетовую карту Газпробанка, через мо..."
1,banki,12901981,None,None,https://www.banki.ru/services/responses/bank/r...,"<p>Поменял паспорт, новый паспорт в банк не по..."
2,banki,12901491,None,None,https://www.banki.ru/services/responses/bank/r...,Оформление и доставка дебетовой карты прошли м...


In [ ]:
domain_stop = {
    "банк","банка","банке","банком","банки",
    "газпромбанк","гпб",
    "клиент","клиента","сотрудник","сотрудники",
    "достоинство","недостаток"
}

ru_stop = {
    "и","в","во","не","что","он","на","я","с","со","как","а","то","все","она","так","его","но","да",
    "ты","к","у","же","вы","за","бы","по","только","ее","мне","было","вот","от","меня","еще","нет",
    "о","из","ему","теперь","когда","даже","ну","вдруг","ли","если","уже","или","ни","быть","был",
    "него","до","вас","нибудь","опять","уж","вам","ведь","там","потом","себя","ничего","ей","может",
    "они","тут","где","есть","надо","ней","для","мы","тебя","их","чем","была","сам","чтоб","без"
}

token_re = re.compile(r"[A-Z]{2,}|[а-яё]{3,}", re.IGNORECASE)

def make_topic_text(text: str) -> str:
    t = normalize_bank_entities(text)
    toks = token_re.findall(t.lower())
    toks = [w for w in toks if w not in ru_stop and w not in domain_stop]
    return " ".join(toks)

df_all["text_topic"] = df_all["text"].apply(make_topic_text)

In [ ]:
BANK_ENTITIES = {
    "CARD": [
        "карта","карты","карту","карточка","карточки"
    ],

    "APP": [
        "приложение","мобильное приложение","онлайн банк",
        "мобайл","app"
    ],

    "SUPPORT": [
        "поддержка","оператор","колл центр",
        "чат поддержки","горячая линия"
    ],

    "PAYMENT": [
        "платеж","платёж","перевод","оплата",
        "транзакция"
    ],

    "ACCOUNT": [
        "счет","счёт","расчетный счет",
        "баланс"
    ],

    "LOAN": [
        "кредит","кредита","ипотека",
        "рассрочка"
    ]
}

In [ ]:
def normalize_bank_entities(text: str):

    text = text.lower()

    for entity, variants in BANK_ENTITIES.items():
        for v in variants:
            text = re.sub(rf"\b{v}\b", entity, text)

    return text

In [ ]:
df_all["text"] = df_all["text"].apply(normalize_bank_entities)

In [ ]:
df_all[["text", "text_topic"]].head(3)

,text,text_topic
0,"оформлял дебетовую CARD газпробанка, через моб...",оформлял дебетовую card газпробанка через моби...
1,"<p>поменял паспорт, новый паспорт в банк не по...",поменял паспорт новый паспорт подавал связи эт...
2,оформление и доставка дебетовой CARD прошли ма...,оформление доставка дебетовой card прошли макс...


In [ ]:
def build_sentiment_model(hf_device: int):
    model_name = "blanchefort/rubert-base-cased-sentiment"
    tok = AutoTokenizer.from_pretrained(model_name)
    mdl = AutoModelForSequenceClassification.from_pretrained(model_name)
    clf = pipeline(
        "text-classification",
        model=mdl,
        tokenizer=tok,
        truncation=True,
        max_length=256,
        device=hf_device,
        top_k=None
    )
    return clf, model_name

def normalize_sentiment_label(label: str) -> str:
    l = label.lower()
    mapping = {
        "label_0": "negative",
        "label_1": "neutral",
        "label_2": "positive",
        "negative": "negative",
        "neutral": "neutral",
        "positive": "positive"
    }
    return mapping.get(l, l)

def run_sentiment(clf, texts: List[str], batch_size: int = 32):
    out = []
    for i in tqdm(range(0, len(texts), batch_size), desc="Sentiment"):
        batch = texts[i:i + batch_size]
        preds = clf(batch)
        if preds and isinstance(preds[0], list):
            preds = [p[0] for p in preds]
        out.extend(preds)
    return out

clf, sent_model_name = build_sentiment_model(HF_DEVICE)
preds = run_sentiment(clf, df_all["text"].tolist(), batch_size=24)

df_all["sentiment_label"] = [normalize_sentiment_label(p["label"]) for p in preds]
df_all["sentiment_score"] = [float(p["score"]) for p in preds]
df_all["sentiment_model"] = sent_model_name

df_all[["source","sentiment_label","sentiment_score","text"]].head(5)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: blanchefort/rubert-base-cased-sentiment
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Sentiment: 100%|██████████| 30/30 [04:28<00:00,  8.96s/it]


,source,sentiment_label,sentiment_score,text
0,banki,neutral,0.764275,"оформлял дебетовую CARD газпробанка, через моб..."
1,banki,positive,0.981153,"<p>поменял паспорт, новый паспорт в банк не по..."
2,banki,positive,0.980701,оформление и доставка дебетовой CARD прошли ма...
3,banki,positive,0.981353,слова благодарности сотруднику банка! <p>приво...
4,banki,positive,0.978063,бесконечная благодарность я стала жертвой моше...


In [ ]:
def build_embedder(torch_device: str):
    model_name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    emb = SentenceTransformer(model_name, device=torch_device)
    return emb, model_name

def get_embeddings(emb, texts: List[str], batch_size: int = 64) -> np.ndarray:
    vecs = emb.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        normalize_embeddings=True
    )
    return np.array(vecs)

emb, emb_model_name = build_embedder(TORCH_DEVICE)
embs = get_embeddings(emb, df_all["text"].tolist(), batch_size=64)

df_all["embedding_model"] = emb_model_name
embs.shape

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

(704, 384)

In [ ]:
def cluster_topics(embeddings: np.ndarray):
    reducer = umap.UMAP(
        n_neighbors=15,
        n_components=5,
        metric="cosine",
        random_state=42
    )
    emb_red = reducer.fit_transform(embeddings)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=25,
        min_samples=10,
        metric="euclidean",
        cluster_selection_method="eom"
    )
    labels = clusterer.fit_predict(emb_red)
    probs = clusterer.probabilities_

    return labels, probs

labels, probs = cluster_topics(embs)
df_all["topic_id"] = labels
df_all["topic_prob"] = probs

df_all["topic_id"].value_counts().head(10)

/usr/local/lib/python3.12/dist-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


topic_id
1    399
0    305
Name: count, dtype: int64

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

def topic_keywords(texts_topic, labels, top_n=12):
    df = pd.DataFrame({"text": texts_topic, "topic": labels})
    df = df[df["topic"] >= 0].copy()

    grouped = df.groupby("topic")["text"].apply(lambda x: " ".join(x)).reset_index()

    vectorizer = TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=3,
        max_df=0.6,
        sublinear_tf=True
    )
    X = vectorizer.fit_transform(grouped["text"])
    vocab = np.array(vectorizer.get_feature_names_out())

    rows = []
    for i, topic_id in enumerate(grouped["topic"].tolist()):
        row = X[i].toarray().ravel()
        top_idx = row.argsort()[::-1][:top_n]
        rows.append({
            "topic": int(topic_id),
            "size": int((labels == topic_id).sum()),
            "keywords": ", ".join(vocab[top_idx].tolist())
        })

    return pd.DataFrame(rows).sort_values("size", ascending=False)

In [ ]:
top_topic = kw.iloc[0]["topic"]
df_all[df_all["topic_id"] == top_topic][["sentiment_label","text"]].head(10)

,sentiment_label,text
9,negative,самый ужасный банк в обслуживании самый ужасны...
10,negative,"граждане, бегите из этого банка, если хотите, ..."
11,negative,как газпромбанк получает млрд на деньгах гражд...
13,positive,выбираю услуги по программе лояльности газпром...
15,negative,премиальное обслуживание много лет являюсь пре...
21,negative,являюсь клиентом газпромбанка много лет и с ка...
23,neutral,"я пользовалась кредитной картой газпромбанка, ..."
31,negative,"улеглись эмоции и прошли праздники, пришло вре..."
32,positive,я являюсь премиальным клиентом газпромбанка. в...
33,negative,доброго времени суток! никому не советую связы...


In [ ]:
df_all.to_csv(OUT_DIR / "reviews_enriched.csv", index=False, encoding="utf-8")
kw.to_csv(OUT_DIR / "topics_keywords.csv", index=False, encoding="utf-8")

print("Saved:", OUT_DIR / "reviews_enriched.csv")
print("Saved:", OUT_DIR / "topics_keywords.csv")